# PerCE — Using Your Own Model

PerCE works with **any classifier** that maps `(N, C, T) → predictions`. This notebook shows how to plug in:

1. A **PyTorch** model (most common for time series)
2. A **scikit-learn** pipeline
3. A **Keras/TensorFlow** model
4. Any **custom callable**

We use a small synthetic dataset so you can run this without downloading anything.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from perce import PerCEExplainer, evaluate_batch

# Synthetic data — 50 instances, 8 channels, 500 time points, binary labels
rng    = np.random.default_rng(0)
N, C, T = 100, 8, 500
X_train = rng.normal(0, 1, (N, C, T)).astype(np.float32)
# Class 1 has higher mean in channel 0 segments 2-4
y_train = (X_train[:, 0, 100:300].mean(axis=1) > 0).astype(int)

X_test  = rng.normal(0, 1, (20, C, T)).astype(np.float32)
y_test  = (X_test[:, 0, 100:300].mean(axis=1) > 0).astype(int)

print(f"Train: {X_train.shape}, labels: {dict(zip(*np.unique(y_train, return_counts=True)))}")

---
## Option A: PyTorch Model

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    class SimpleTSNet(nn.Module):
        def __init__(self, n_channels, n_classes=2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(n_channels, 32, 7, padding=3), nn.ReLU(),
                nn.Conv1d(32, 64, 5, padding=2),         nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(64, n_classes),
                nn.Softmax(dim=1)
            )
        def forward(self, x): return self.net(x)

    device = torch.device("cpu")
    pt_net = SimpleTSNet(n_channels=C).to(device)

    # Quick training loop
    opt = torch.optim.Adam(pt_net.parameters(), lr=1e-3)
    Xt  = torch.tensor(X_train); yt = torch.tensor(y_train, dtype=torch.long)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=16, shuffle=True)

    pt_net.train()
    for epoch in range(15):
        for xb, yb in loader:
            opt.zero_grad()
            nn.CrossEntropyLoss()(pt_net(xb), yb).backward()
            opt.step()

    # ── PerCE wrapper ────────────────────────────────────────────────────
    def pytorch_model(X_np: np.ndarray) -> np.ndarray:
        """Wrap a PyTorch model for PerCE. Returns (N, n_classes) array."""
        pt_net.eval()
        with torch.no_grad():
            out = pt_net(torch.tensor(X_np, dtype=torch.float32)).numpy()
        return out   # (N, 2) softmax probabilities

    # Test it
    preds = pytorch_model(X_test[:3])
    print("PyTorch model — sample output:", preds)

    # ── Run PerCE ────────────────────────────────────────────────────────
    exp_pt = PerCEExplainer(model=pytorch_model, n_segments=5, k=3)
    exp_pt.fit(X_train, y_train)
    result_pt = exp_pt.explain(X_test[0], target_class=1 - y_test[0])
    print("\n" + result_pt.summary())

except ImportError:
    print("PyTorch not installed — skipping. pip install torch")

---
## Option B: scikit-learn Pipeline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# sklearn expects 2D input — flatten the time series
X_train_flat = X_train.reshape(N, -1)
X_test_flat  = X_test.reshape(20, -1)

sk_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(n_estimators=50, random_state=0))
])
sk_pipe.fit(X_train_flat, y_train)

sk_acc = sk_pipe.score(X_test_flat, y_test)
print(f"sklearn RF accuracy: {sk_acc:.2%}")

# ── PerCE wrapper ────────────────────────────────────────────────────────
def sklearn_model(X_np: np.ndarray) -> np.ndarray:
    """Wrap any sklearn pipeline for PerCE."""
    X_flat = X_np.reshape(len(X_np), -1)
    return sk_pipe.predict_proba(X_flat)   # (N, 2)

# ── Run PerCE ────────────────────────────────────────────────────────────
exp_sk = PerCEExplainer(model=sklearn_model, n_segments=5, k=3)
exp_sk.fit(X_train, y_train)
result_sk = exp_sk.explain(X_test[1], target_class=1 - int(y_test[1]))
print("\n" + result_sk.summary())

---
## Option C: Keras / TensorFlow

In [ ]:
try:
    import tensorflow as tf

    # Build a simple 1D CNN
    tf_model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu",
                               input_shape=(T, C)),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(2, activation="softmax")
    ])
    tf_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                     metrics=["accuracy"])

    # Keras expects (N, T, C) — transpose before training
    X_train_tf = X_train.transpose(0, 2, 1)
    X_test_tf  = X_test.transpose( 0, 2, 1)

    tf_model.fit(X_train_tf, y_train, epochs=10, batch_size=16, verbose=0)
    print(f"Keras model accuracy: {tf_model.evaluate(X_test_tf, y_test, verbose=0)[1]:.2%}")

    # ── PerCE wrapper ────────────────────────────────────────────────────
    def keras_model(X_np: np.ndarray) -> np.ndarray:
        """Wrap a Keras model. Note: Keras expects (N, T, C), PerCE passes (N, C, T)."""
        X_transposed = X_np.transpose(0, 2, 1)   # (N, C, T) → (N, T, C)
        return tf_model.predict(X_transposed, verbose=0)   # (N, 2)

    exp_tf = PerCEExplainer(model=keras_model, n_segments=5, k=3)
    exp_tf.fit(X_train, y_train)
    result_tf = exp_tf.explain(X_test[2], target_class=1 - int(y_test[2]))
    print("\n" + result_tf.summary())

except ImportError:
    print("TensorFlow not installed — skipping. pip install tensorflow")

---
## Batch Evaluation

In [ ]:
# Using the sklearn model as example
results = exp_sk.explain_batch(
    X_test,
    target_classes=[1 - int(y) for y in y_test],
    verbose=True
)

summary = evaluate_batch(results)
print(f"\nValidity rate : {summary['validity_rate']:.2%}")
print(f"Proximity     : {summary['proximity_mean']:.4f} ± {summary['proximity_std']:.4f}")
print(f"Sparsity      : {summary['sparsity_mean']:.4f} ± {summary['sparsity_std']:.4f}")
print(f"Diversity     : {summary['diversity']:.4f}")

---
## Key Tips

| Situation | Recommendation |
|-----------|---------------|
| Model returns class indices `(N,)` | Wrap to return `(N, K)` probabilities |
| Model expects different input shape | Add a transpose inside the wrapper |
| Very long sequences (T > 10,000) | Reduce `n_segments` or use `dtw_window=0.05` |
| Many channels (C > 20) | Consider increasing `n_importance_samples` |
| Binary sigmoid output | Return `np.stack([1-p, p], axis=1)` |

The rule is simple: **make the model callable accept `(N, C, T)` numpy arrays and return `(N, K)` class probabilities**.